# OpenSearch: how the Knowledge Base searches

Local cluster from `docker-compose.yml` (security plugin disabled, no auth, published on port 9201). The app searches the `items` index.

In [1]:
from opensearchpy import OpenSearch

client = OpenSearch("http://localhost:9201")
print(client.info()["version"]["number"])

3.9.0


## 1 · Keyword search (BM25)

The app's query matches the words in title (weighted ×2), summary, key points, a Thought's text and the Owner's note.

In [ ]:
# QUERY = "dron"
QUERY = "formula"

keyword_query = {
    "multi_match": {"query": QUERY, "fields": ["title^2", "summary", "key_points", "text", "annotations.notes"]}
}
keyword_resp = client.search(
    index="items",
    # body={"query": keyword_query, "_source": ["title", "normalised_url", "tags"], "size": 20},  # skip the big embedding vector
    # body={"query": keyword_query, "_source": ["title", "normalised_url", "tags", "embedding"], "size": 20}, # show also embedding 
    body={"query": keyword_query, "_source": ["title", "normalised_url", "tags", "summary", "key_points", "text"], "size": 20} # show text
)


def show(resp):
    for hit in resp["hits"]["hits"][:5]:
        print(round(hit["_score"], 3), hit["_source"]["title"])


show(keyword_resp)

2.375 ‘It’s taken a while!’ – Verstappen thrilled with maiden Singapore win at 10th attempt
2.224 SINGAPORE LOWDOWN: All the key moments as Max Verstappen dominates and Ferrari face strategy turmoil


In [3]:
keyword_resp["hits"]["hits"][0]

{'_index': 'items',
 '_id': '0a75631e99d17614de3623d0a604ae88bfd5cf9983ce35144a5376e23c9145aa',
 '_score': 2.3751194,
 '_source': {'summary': "Max Verstappen achieved his first victory at the Singapore Grand Prix after nine previous attempts, marking a significant milestone in his career. He had previously secured four podium finishes at the Marina Bay Street Circuit, including two second-place finishes in 2024 and 2025. This win came during his 250th Grand Prix start, where he demonstrated strong performance throughout the race, particularly in defending against Charles Leclerc at the start. Verstappen's victory adds to Red Bull's recent success, following their first win of the 2026 season in Bahrain. He expressed satisfaction with the team's progress and motivation to continue delivering strong results.",
  'normalised_url': 'https://www.formula1.com/en/latest/article/its-taken-a-while-verstappen-thrilled-with-maiden-singapore-win-at-10th-attempt.7A6wicGmDmoONtJzg1TdLl',
  'title': 

## 2 · Vector search (k-NN)

The query is embedded by the same model as the Items (1536 numbers), and OpenSearch returns the nearest Items by cosine similarity.

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
vector = OpenAI().embeddings.create(model="text-embedding-3-small", input=QUERY).data[0].embedding  # paid call
print(len(vector), vector[:3])

knn_query = {"knn": {"embedding": {"vector": vector, "k": 50}}}
vector_resp = client.search(index="items", body={"query": knn_query, "_source": ["title"], "size": 20})
show(vector_resp)

1536 [0.01023101806640625, -0.0190277099609375, -0.01268768310546875]
0.678 SINGAPORE LOWDOWN: All the key moments as Max Verstappen dominates and Ferrari face strategy turmoil
0.672 Verstappen si v Singapuru vyjel jubilejní padesátou pole position v kariéře
0.666 První triumf Maxe Verstappena v Singapuru. Lewis Hamilton doplatil na strategii
0.662 ‘It’s taken a while!’ – Verstappen thrilled with maiden Singapore win at 10th attempt
0.635 Verstappen si naprosto podrobil Singapur a vyhrál zde poprvé v kariéře


In [12]:
# vector

## 3 · Hybrid search

Both queries run at once. The `kb-hybrid` search pipeline scales each result list to 0–1 (min-max) and averages them 50/50.

In [6]:
client.transport.perform_request("GET", "/_search/pipeline/kb-hybrid") # mame nadefinovane pri zakladani indexu, funguje na urovni db.

{'kb-hybrid': {'description': 'kb hybrid search: min-max normalised BM25 + k-NN, weighted arithmetic mean',
  'phase_results_processors': [{'normalization-processor': {'normalization': {'technique': 'min_max'},
     'combination': {'technique': 'arithmetic_mean',
      'parameters': {'weights': [0.5, 0.5]}}}}]}}

In [7]:
hybrid_resp = client.search(
    index="items",
    body={"query": {"hybrid": {"queries": [keyword_query, knn_query]}}, "_source": ["title"], "size": 5},
    params={"search_pipeline": "kb-hybrid"},
)


def rank(resp, item_id):
    ids = [hit["_id"] for hit in resp["hits"]["hits"]]
    return ids.index(item_id) + 1 if item_id in ids else "–"


print("hybrid  keyword  vector  title")
for hit in hybrid_resp["hits"]["hits"]:
    print(f"{hit['_score']:.3f}  {rank(keyword_resp, hit['_id'])!s:>7}  {rank(vector_resp, hit['_id'])!s:>6}  {hit['_source']['title']}")

hybrid  keyword  vector  title
0.960        1       4  ‘It’s taken a while!’ – Verstappen thrilled with maiden Singapore win at 10th attempt
0.500        2       1  SINGAPORE LOWDOWN: All the key moments as Max Verstappen dominates and Ferrari face strategy turmoil
0.486        –       2  Verstappen si v Singapuru vyjel jubilejní padesátou pole position v kariéře
0.470        –       3  První triumf Maxe Verstappena v Singapuru. Lewis Hamilton doplatil na strategii
0.389        –       5  Verstappen si naprosto podrobil Singapur a vyhrál zde poprvé v kariéře


## 4 · How the data are stored

`items` is the current view of each Item (what is searched); `versions` keeps every Capture, never updated.

In [13]:
# mapping = client.indices.get_mapping(index="items")["items"]["mappings"]["properties"]
# {field: spec.get("type", "object") for field, spec in mapping.items()}, mapping["embedding"]

In [14]:
item_id = keyword_resp["hits"]["hits"][0]["_id"]  # sha256 of the normalised URL
item = client.get(index="items", id=item_id)["_source"]
item["embedding"] = f"{len(item['embedding'])} floats {item['embedding'][:3]} …"
item

{'id': '0a75631e99d17614de3623d0a604ae88bfd5cf9983ce35144a5376e23c9145aa',
 'normalised_url': 'https://www.formula1.com/en/latest/article/its-taken-a-while-verstappen-thrilled-with-maiden-singapore-win-at-10th-attempt.7A6wicGmDmoONtJzg1TdLl',
 'original_url': 'https://www.formula1.com/en/latest/article/its-taken-a-while-verstappen-thrilled-with-maiden-singapore-win-at-10th-attempt.7A6wicGmDmoONtJzg1TdLl',
 'media_type': 'article',
 'first_captured_at': '2026-10-12T09:23:15.987000Z',
 'captured_at': '2026-10-12T09:23:15.987000Z',
 'current_version_id': '0a75631e99d17614de3623d0a604ae88bfd5cf9983ce35144a5376e23c9145aa:2026-10-12T09:23:15.987Z',
 'title': '‘It’s taken a while!’ – Verstappen thrilled with maiden Singapore win at 10th attempt',
 'summary': "Max Verstappen achieved his first victory at the Singapore Grand Prix after nine previous attempts, marking a significant milestone in his career. He had previously secured four podium finishes at the Marina Bay Street Circuit, including

In [10]:
client.search(
    index="versions",
    body={"query": {"term": {"item_id": item_id}}, "_source": {"excludes": ["enrichment.embedding"]}},
)["hits"]["hits"]

[{'_index': 'versions',
  '_id': '0a75631e99d17614de3623d0a604ae88bfd5cf9983ce35144a5376e23c9145aa:2026-10-12T09:23:15.987Z',
  '_score': 1.0,
  '_source': {'enrichment': {'summary': "Max Verstappen achieved his first victory at the Singapore Grand Prix after nine previous attempts, marking a significant milestone in his career. He had previously secured four podium finishes at the Marina Bay Street Circuit, including two second-place finishes in 2024 and 2025. This win came during his 250th Grand Prix start, where he demonstrated strong performance throughout the race, particularly in defending against Charles Leclerc at the start. Verstappen's victory adds to Red Bull's recent success, following their first win of the 2026 season in Bahrain. He expressed satisfaction with the team's progress and motivation to continue delivering strong results.",
    'embedding_model': 'text-embedding-3-small',
    'taxonomy_version': 10,
    'language': 'en',
    'model': 'gpt-4o-mini',
    'title':